# Đánh giá marketing — so sánh model (Qwen2b / Qwen7b / GemmaE2B / GemmaE4B)

Tái sử dụng **cùng metric** như `marketing_finetune_evaluation.ipynb` (Faithfulness + Expansion + Marketing Vibe).

**Dữ liệu:** `dataset/high_quality_mock_cases_100.json` (cột `actual_output` = bài model sinh).

**Model sinh bài:** server local tương thích OpenAI — `LOCAL_LLM_BACKEND` (`lmstudio` / `ollama`). **LOCAL_MODEL_ID** + **LOCAL_OPENAI_BASE** tự gán từ `LMSTUDIO_*` hoặc `OLLAMA_*` tùy backend.

**Judge (DeepEval):** `OPENAI_API_KEY` **hoặc** `AZURE_OPENAI_ENDPOINT` + `AZURE_OPENAI_API_KEY`. Mặc định model judge: `gpt-5.4-mini` / `DEEPEVAL_JUDGE_MODEL`. Không dùng LM Studio làm judge.

**Mỗi lần chạy:** CSV tại `results/runs/…`, tổng hợp (append) tại `dataset/high_quality_mock_cases_100.eval_summary.csv`.

In [ ]:
# %pip install -q deepeval openai pandas python-dotenv

In [25]:
import os
from pathlib import Path
from typing import Optional

from dotenv_reload import reload_dotenv

_env_path = reload_dotenv()
ROOT = _env_path.parent.resolve() if _env_path else Path.cwd().resolve()

# ===== Hyperparameters — chỉnh tập trung tại đây =====

# Đường dẫn dữ liệu & output (tương đối ROOT)
DATASET_REL = "dataset/high_quality_mock_cases_100.json"
SUMMARY_REL = "results/high_quality_mock_cases_100.eval_summary.csv"
RUNS_REL = "results/runs"

# CSV ghi đè output theo case_id (None = chỉ dùng actual_output trong JSON)
OUTPUT_FROM_CSV: Optional[Path] = None

# Judge: OPENAI_API_KEY hoặc AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY
def _env(k: str) -> str:
    return (os.getenv(k) or "").strip()


JUDGE_MODEL = os.getenv("OPENAI_MODEL")
has_o, has_a = bool(_env("OPENAI_API_KEY")), bool(_env("AZURE_OPENAI_ENDPOINT") and _env("AZURE_OPENAI_API_KEY"))
if has_o and has_a:
    raise RuntimeError("Chỉ một: OPENAI_API_KEY hoặc cặp AZURE_OPENAI_* cho judge.")
if not has_o and not has_a:
    raise RuntimeError("Judge cần OPENAI_API_KEY hoặc AZURE_OPENAI_ENDPOINT + AZURE_OPENAI_API_KEY.")
if has_a:
    os.environ.setdefault("OPENAI_API_VERSION", "2024-08-01-preview")
    os.environ["USE_AZURE_OPENAI"] = "true"
    os.environ.pop("USE_OPENAI_MODEL", None)
    JUDGE_BACKEND = "azure"
else:
    os.environ.pop("USE_AZURE_OPENAI", None)
    JUDGE_BACKEND = "openai"
os.environ["OPENAI_MODEL"] = JUDGE_BACKEND

# Backend sinh bài (OpenAI-compatible): "lmstudio" | "ollama" — đặt trong .env: LOCAL_LLM_BACKEND=ollama
LOCAL_LLM_BACKEND = os.getenv("LOCAL_LLM_BACKEND", "lmstudio").strip().lower()
if LOCAL_LLM_BACKEND not in ("lmstudio", "ollama"):
    raise ValueError('LOCAL_LLM_BACKEND phải là "lmstudio" hoặc "ollama".')

# LM Studio / Ollama — LOCAL_MODEL_ID / LOCAL_OPENAI_BASE / LOCAL_API_KEY gán sau theo LOCAL_LLM_BACKEND
LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://127.0.0.1:1234/v1").rstrip("/")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434/v1").rstrip("/")

GENERATION_TEMPERATURE = 0.4

# Tiến trình đánh giá: in và đo thời gian mỗi N case (1–N, N+1–2N, …)
EVAL_PROGRESS_CHUNK = 10

DATASET_PATH = ROOT / DATASET_REL
SUMMARY_PATH = ROOT / SUMMARY_REL
RUNS_DIR = ROOT / RUNS_REL


def _to_openai_v1_base(url: str) -> str:
    b = url.rstrip("/")
    return b if b.endswith("/v1") else f"{b}/v1"


if LOCAL_LLM_BACKEND == "lmstudio":
    LOCAL_OPENAI_BASE = _to_openai_v1_base(LMSTUDIO_BASE_URL)
    LOCAL_MODEL_ID = (os.getenv("LMSTUDIO_MODEL_ID") or "local-model").strip() or "local-model"
    LOCAL_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")
elif LOCAL_LLM_BACKEND == "ollama":
    LOCAL_OPENAI_BASE = _to_openai_v1_base(OLLAMA_BASE_URL)
    LOCAL_MODEL_ID = (os.getenv("OLLAMA_MODEL_ID") or "llama3.2").strip() or "llama3.2"
    LOCAL_API_KEY = os.getenv("OLLAMA_API_KEY", "ollama")

print("JUDGE_MODEL:", JUDGE_MODEL, f"({JUDGE_BACKEND})")
print(
    "Sinh bài:",
    LOCAL_LLM_BACKEND,
    "| OpenAI-compatible:",
    LOCAL_OPENAI_BASE,
    "| LOCAL_MODEL_ID:",
    LOCAL_MODEL_ID,
)

JUDGE_MODEL: md-gpt-5.4-mini (azure)
Sinh bài: ollama | OpenAI-compatible: http://192.168.92.26:11434/v1 | LOCAL_MODEL_ID: qwen3.5-2b-marketing


In [15]:
# Verify nhanh: Judge (OpenAI/Azure) + local LLM (LM Studio / Ollama) — phải chạy sau ô hyperparameters
from openai import AzureOpenAI, OpenAI

_hello_prompt = "Xin chào"

_judge = AzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/"),
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ.get("OPENAI_API_VERSION", "2024-08-01-preview"),
)
_judge_model_id = JUDGE_MODEL


_r = _judge.chat.completions.create(
    model=_judge_model_id,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Judge:", (_r.choices[0].message.content or "").strip())

Judge: Xin chào! Mình có thể giúp gì cho bạn hôm nay?


In [17]:
_local = OpenAI(base_url=LOCAL_OPENAI_BASE, api_key=LOCAL_API_KEY)
_r2 = _local.chat.completions.create(
    model=LOCAL_MODEL_ID,
    messages=[{"role": "user", "content": _hello_prompt}],
)
print("Local:", (_r2.choices[0].message.content or "").strip())

Local: <think>

</think>

Chào em nhé! Rất vui được làm việc cùng team Marketing/Social Media. 

Nếu các em cần hỗ trợ với chiến dịch Facebook marketing trên Facebook, TikTok Ads, content post, caption tiếng Việt/Văn Phạn, hoặc lên kế hoạch KPI, hãy nêu rõ mục tiêu, đối tượng và ngân sách để em tư vấn ngay nhé! 😊


In [18]:
import json
import math
import os
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Type

import numpy as np
import pandas as pd
from deepeval.metrics import FaithfulnessMetric, GEval
from deepeval.models import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

from dotenv_reload import reload_dotenv

reload_dotenv()
_o = (os.getenv("OPENAI_API_KEY") or "").strip()
_az = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip() and (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
if not (_o or _az):
    raise RuntimeError("Chạy ô hyperparameters trước hoặc đặt OPENAI_API_KEY hoặc AZURE_OPENAI_* trong .env.")
print("DeepEval metrics: sẵn sàng (judge =", os.environ.get("OPENAI_MODEL", "?"), ").")


class JudgeLLM(DeepEvalBaseLLM):
    """Bọc _judge client (OpenAI / AzureOpenAI) thành DeepEval-compatible model.

    Truyền instance này vào metrics để DeepEval dùng client của mình
    thay vì tự khởi tạo lại qua initialize_model().
    """

    def __init__(self, client, model_id: str) -> None:
        self._client = client
        self._model_id = model_id

    def get_model_name(self) -> str:
        return self._model_id

    def load_model(self):
        return self._client

    def generate(self, prompt: str, schema: Optional[Type] = None):
        # DeepEval (using_native_model=False): phải trả về str hoặc instance schema —
        # KHÔNG trả (text, cost); tuple chỉ dùng với native DeepEval models.
        if schema is not None:
            try:
                resp = self._client.beta.chat.completions.parse(
                    model=self._model_id,
                    messages=[{"role": "user", "content": prompt}],
                    response_format=schema,
                )
                return resp.choices[0].message.parsed
            except Exception:
                resp = self._client.chat.completions.create(
                    model=self._model_id,
                    messages=[{"role": "user", "content": prompt}],
                    response_format={"type": "json_object"},
                )
                raw = (resp.choices[0].message.content or "{}").strip()
                return schema.model_validate_json(raw)
        resp = self._client.chat.completions.create(
            model=self._model_id,
            messages=[{"role": "user", "content": prompt}],
        )
        return (resp.choices[0].message.content or "").strip()

    async def a_generate(self, prompt: str, schema: Optional[Type] = None):
        return self.generate(prompt, schema)


judge_llm = JudgeLLM(_judge, _judge_model_id)
print(f"JudgeLLM: {judge_llm.get_model_name()}")

JUDGE_KWARGS = dict(model=judge_llm, async_mode=False, verbose_mode=False)

PRICE_RE = re.compile(
    r"(?:\d{1,3}(?:[.,]\d{3})+|\d+(?:[.,]\d+)?)\s*(?:VNĐ|vnd|đ|d|USD|\$|usd|triệu|tr|k)\b",
    re.IGNORECASE,
)
SPEC_RE = re.compile(
    r"\b\d+(?:[.,]\d+)?\s*(?:GB|gb|TB|tb|mAh|mah|W|w|V|v|Hz|hz|inch|\"|cm|mm|kg|g|ml|l|%)\b",
    re.IGNORECASE,
)
QUOTED_RE = re.compile(r"[\"\"]([^\"\"]{2,80})[\"\"]")


def extract_candidate_entities(seed: str) -> List[str]:
    found: List[str] = []
    for m in PRICE_RE.finditer(seed):
        found.append(m.group(0).strip())
    for m in SPEC_RE.finditer(seed):
        found.append(m.group(0).strip())
    for m in QUOTED_RE.finditer(seed):
        found.append(m.group(1).strip())
    for line in seed.replace(";", "\n").split("\n"):
        if ":" in line:
            key, val = line.split(":", 1)
            if re.search(r"sản phẩm|model|tên", key, re.I) and len(val.strip()) >= 2:
                found.append(val.strip().split(".")[0].strip())
    seen = set()
    out = []
    for e in found:
        k = e.lower()
        if k not in seen and len(e) >= 2:
            seen.add(k)
            out.append(e)
    return out


def entity_presence_score(seed: str, output: str) -> Tuple[float, Dict[str, Any]]:
    entities = extract_candidate_entities(seed)
    if not entities:
        return 1.0, {"entities": [], "matched": [], "note": "no entities"}
    out_lower = output.lower()
    matched = [e for e in entities if e.lower() in out_lower]
    score = len(matched) / len(entities)
    return score, {"entities": entities, "matched": matched}


def extract_first_hook(markdown_text: str, max_chars: int = 400) -> str:
    text = markdown_text.strip()
    if not text:
        return ""
    lines = text.splitlines()
    start = 0
    if lines and lines[0].lstrip().startswith("#"):
        start = 1
    body = "\n".join(lines[start:]).strip()
    parts = re.split(r"\n\n+", body, maxsplit=1)
    first_block = parts[0] if parts else body
    if len(first_block) > max_chars:
        return first_block[:max_chars].rstrip() + "…"
    return first_block


CTA_PATTERNS = [
    r"\b(mua ngay|đặt hàng|inbox|liên hệ|đăng ký|nhận tư vấn|thử ngay|khám phá ngay)\b",
    r"[👉✨🔥]\s*\*?\*?[A-Za-zÀ-ỹ0-9]",
    r"https?://",
]


def rule_markdown_structure(md: str) -> Tuple[float, Dict[str, Any]]:
    has_h1 = bool(re.search(r"(?m)^#\s+\S", md))
    has_h2 = bool(re.search(r"(?m)^##\s+\S", md))
    bullets = len(re.findall(r"(?m)^\s*[-*]\s+\S", md))
    has_bullet = bullets >= 1
    checks = {"h1": has_h1, "h2": has_h2, "bullet_lines": bullets}
    score = sum([has_h1, has_h2, has_bullet]) / 3.0
    return score, checks


def rule_cta_end(md: str, tail_chars: int = 600) -> Tuple[float, Dict[str, Any]]:
    tail = md[-tail_chars:].lower()
    hit = any(re.search(p, tail, re.I) for p in CTA_PATTERNS)
    return (1.0 if hit else 0.0), {"cta_found": hit, "tail_preview": tail[-120:]}


@dataclass
class FaithfulnessResult:
    rule_entity_score: float
    rule_detail: Dict[str, Any]
    llm_faithfulness_score: float
    llm_reason: str = ""
    combined_score: float = 0.0

    def __post_init__(self):
        self.combined_score = 0.5 * self.rule_entity_score + 0.5 * self.llm_faithfulness_score


class FaithfulnessEvaluator:
    def __init__(self) -> None:
        self.metric = FaithfulnessMetric(threshold=0.5, **JUDGE_KWARGS)

    def evaluate_one(self, input_title: str, seed_content: str, actual_output: str) -> FaithfulnessResult:
        rule_score, rule_detail = entity_presence_score(seed_content, actual_output)
        case = LLMTestCase(
            input=input_title,
            actual_output=actual_output,
            retrieval_context=[seed_content],
        )
        self.metric.measure(case)
        llm_score = float(self.metric.score or 0.0)
        reason = getattr(self.metric, "reason", "") or ""
        return FaithfulnessResult(
            rule_entity_score=rule_score,
            rule_detail=rule_detail,
            llm_faithfulness_score=llm_score,
            llm_reason=reason,
        )


@dataclass
class ExpansionResult:
    llm_expansion_score: float
    llm_reason: str
    rule_length_score: float
    combined_score: float


class ExpansionQualityEvaluator:
    def __init__(self) -> None:
        self.metric = GEval(
            name="Expansion Quality",
            criteria=(
                "Đánh giá khả năng mở rộng từ nội dung mồi thành một bài viết marketing hoàn chỉnh. "
                "Bài phải diễn giải rõ lợi ích cho khách hàng (benefits) và đưa ra ngữ cảnh sử dụng thực tế "
                "dựa trên các tính năng/thông tin thô trong phần context (nội dung mồi). "
                "Trừ điểm nếu chỉ lặp lại bullet kỹ thuật mà không giải thích giá trị."
            ),
            evaluation_params=[
                LLMTestCaseParams.INPUT,
                LLMTestCaseParams.ACTUAL_OUTPUT,
                LLMTestCaseParams.CONTEXT,
            ],
            **JUDGE_KWARGS,
        )

    def _length_score(self, seed: str, output: str) -> float:
        ratio = (len(output.strip()) + 1) / (len(seed.strip()) + 1)
        if ratio >= 3.0:
            return 1.0
        if ratio >= 2.0:
            return 0.7
        if ratio >= 1.2:
            return 0.4
        return 0.1

    def evaluate_one(self, input_title: str, seed_content: str, actual_output: str) -> ExpansionResult:
        case = LLMTestCase(
            input=input_title,
            actual_output=actual_output,
            context=[seed_content],
        )
        self.metric.measure(case)
        llm_s = float(self.metric.score or 0.0)
        reason = getattr(self.metric, "reason", "") or ""
        rlen = self._length_score(seed_content, actual_output)
        combined = 0.9 * llm_s + 0.1 * rlen
        return ExpansionResult(
            llm_expansion_score=llm_s,
            llm_reason=reason,
            rule_length_score=rlen,
            combined_score=combined,
        )


@dataclass
class MarketingVibeResult:
    llm_hook_tone_score: float
    llm_reason: str
    rule_structure_score: float
    rule_cta_score: float
    rule_combined: float
    combined_score: float


class MarketingVibeEvaluator:
    def __init__(self) -> None:
        self.metric = GEval(
            name="Marketing Hook Tone",
            evaluation_steps=[
                "Chỉ đánh giá nội dung sau nhãn [HOOK ĐỂ CHẤM] trong actual_output (bỏ qua phần tiêu đề phụ).",
                "Đánh giá sự lôi cuốn, chuyên nghiệp và khả năng giữ chân người đọc của hook.",
                "So sánh hook với INPUT (yêu cầu/tiêu đề) để xem có mở bài hấp dẫn và đúng hướng không.",
                "Trừ điểm nếu hook quá ngắn, chung chung, hoặc giọng không phù hợp marketing.",
            ],
            evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
            **JUDGE_KWARGS,
        )

    def evaluate_one(self, input_title: str, seed_content: str, actual_output: str) -> MarketingVibeResult:
        hook = extract_first_hook(actual_output)
        hook_block = f"[HOOK ĐỂ CHẤM]\n{hook}\n\n[Tiêu đề bài]\n{input_title}"
        case = LLMTestCase(
            input="Đánh giá hook cho bài marketing sau (kèm tiêu đề).",
            actual_output=hook_block,
        )
        self.metric.measure(case)
        llm_s = float(self.metric.score or 0.0)
        reason = getattr(self.metric, "reason", "") or ""
        s_md, _ = rule_markdown_structure(actual_output)
        s_cta, _ = rule_cta_end(actual_output)
        rule_mix = 0.5 * s_md + 0.5 * s_cta
        combined = 0.7 * llm_s + 0.3 * rule_mix
        return MarketingVibeResult(
            llm_hook_tone_score=llm_s,
            llm_reason=reason,
            rule_structure_score=s_md,
            rule_cta_score=s_cta,
            rule_combined=rule_mix,
            combined_score=combined,
        )


def run_batch(
    cases: Sequence[Dict[str, str]],
    progress_chunk: int = 10,
) -> pd.DataFrame:
    f_ev = FaithfulnessEvaluator()
    e_ev = ExpansionQualityEvaluator()
    m_ev = MarketingVibeEvaluator()
    rows: List[Dict[str, Any]] = []
    n = len(cases)
    pc = max(1, int(progress_chunk))
    chunk_t0: Optional[float] = None
    chunk_start_i = 1

    for i, c in enumerate(cases, start=1):
        if (i - 1) % pc == 0:
            chunk_t0 = time.perf_counter()
            chunk_start_i = i

        title, seed, out = c["input_title"], c["seed_content"], c["actual_output"]
        fr = f_ev.evaluate_one(title, seed, out)
        er = e_ev.evaluate_one(title, seed, out)
        mr = m_ev.evaluate_one(title, seed, out)
        rows.append(
            {
                "case_id": c.get("case_id", ""),
                "faithfulness_rule": fr.rule_entity_score,
                "faithfulness_llm": fr.llm_faithfulness_score,
                "faithfulness_combined": fr.combined_score,
                "expansion_llm": er.llm_expansion_score,
                "expansion_rule_length": er.rule_length_score,
                "expansion_combined": er.combined_score,
                "vibe_llm_hook": mr.llm_hook_tone_score,
                "vibe_rule_md_cta": mr.rule_combined,
                "vibe_combined": mr.combined_score,
                "faithfulness_llm_reason": (fr.llm_reason or "")[:500],
                "expansion_llm_reason": (er.llm_reason or "")[:500],
                "vibe_llm_reason": (mr.llm_reason or "")[:500],
            }
        )

        if chunk_t0 is not None and ((i % pc == 0) or (i == n)):
            elapsed = time.perf_counter() - chunk_t0
            print(
                f"Chunk {chunk_start_i}–{i}/{n} hoàn thành trong {elapsed:.1f}s",
                flush=True,
            )

    return pd.DataFrame(rows)

DeepEval metrics: sẵn sàng (judge = md-gpt-5.4-mini ).
JudgeLLM: md-gpt-5.4-mini


## Load dataset

Hyperparameter ở **ô đầu** (sau `%pip`). ~**3 lời gọi judge / case**. Tổng hợp: **CSV** `eval_summary.csv`.

In [19]:
def load_cases(json_path: Path, override_csv: Optional[Path]) -> List[Dict[str, str]]:
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("Dataset phải là mảng JSON.")
    cases = []
    for row in data:
        cases.append(
            {
                "case_id": str(row["case_id"]),
                "input_title": str(row["input_title"]),
                "seed_content": str(row["seed_content"]),
                "actual_output": str(row.get("actual_output", "")),
            }
        )
    if override_csv and override_csv.is_file():
        df_o = pd.read_csv(override_csv)
        if "case_id" not in df_o.columns or "actual_output" not in df_o.columns:
            raise ValueError("CSV override cần cột: case_id, actual_output")
        m = dict(zip(df_o["case_id"].astype(str), df_o["actual_output"].astype(str)))
        for c in cases:
            if c["case_id"] in m:
                c["actual_output"] = m[c["case_id"]]
    return cases


cases = load_cases(DATASET_PATH, OUTPUT_FROM_CSV)
print(f"Cases: {len(cases)} | dataset: {DATASET_PATH.name} | LOCAL_MODEL_ID: {LOCAL_MODEL_ID}")

Cases: 100 | dataset: high_quality_mock_cases_100.json | LOCAL_MODEL_ID: qwen3.5-2b-marketing


### (Tùy chọn) Sinh `actual_output` qua LM Studio hoặc Ollama

Chạy **sau** ô load `cases`, **trước** ô `run_batch`. Client sinh bài: `LOCAL_OPENAI_BASE`, `LOCAL_MODEL_ID`, `LOCAL_API_KEY` (tự gán theo `LOCAL_LLM_BACKEND`) từ ô hyperparameters đầu tiên.

In [20]:
# Uncomment để sinh bài qua local OpenAI-compatible (LM Studio hoặc Ollama — xem LOCAL_LLM_BACKEND)
# from openai import OpenAI
#
# _gen = OpenAI(base_url=LOCAL_OPENAI_BASE, api_key=LOCAL_API_KEY)
#
# def build_user_prompt(title: str, seed: str) -> str:
#     return f"Viết bài marketing Markdown từ tiêu đề và mồi:\nTiêu đề: {title}\nMồi: {seed}"
#
# for c in cases:
#     r = _gen.chat.completions.create(
#         model=LOCAL_MODEL_ID,
#         messages=[{"role": "user", "content": build_user_prompt(c["input_title"], c["seed_content"])}],
#         temperature=GENERATION_TEMPERATURE,
#     )
#     c["actual_output"] = (r.choices[0].message.content or "").strip()
print("(Khung sinh bài — đang tắt; bỏ comment khi cần.)")

(Khung sinh bài — đang tắt; bỏ comment khi cần.)


In [21]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUNS_DIR.mkdir(parents=True, exist_ok=True)

safe_model = re.sub(r"[^A-Za-z0-9._-]+", "_", LOCAL_MODEL_ID).strip("_") or "model"
csv_path = RUNS_DIR / f"{safe_model}_{RUN_ID}.csv"

df_scores = run_batch(cases, progress_chunk=EVAL_PROGRESS_CHUNK)

base = pd.DataFrame([{**c} for c in cases])
# df_scores already contains case_id — drop it to avoid duplicate columns after concat
df_out = pd.concat(
    [base.reset_index(drop=True), df_scores.drop(columns=["case_id"]).reset_index(drop=True)],
    axis=1,
)
df_out.insert(0, "run_id", RUN_ID)
df_out.insert(1, "judge_backend", JUDGE_BACKEND)
df_out.insert(2, "judge_model", JUDGE_MODEL)
df_out.insert(3, "local_llm_backend", LOCAL_LLM_BACKEND)
df_out.insert(4, "local_model_id", LOCAL_MODEL_ID)
df_out.insert(5, "local_openai_base", LOCAL_OPENAI_BASE)

df_out.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"Đã lưu: {csv_path}")

c:\Users\vqnhan\AppData\Local\Programs\Python\Python314\Lib\site-packages\rich\live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Chunk 1–10/100 hoàn thành trong 120.8s


Chunk 11–20/100 hoàn thành trong 107.3s


Chunk 21–30/100 hoàn thành trong 106.6s


Chunk 31–40/100 hoàn thành trong 112.1s


...

Chunk 51–60/100 hoàn thành trong 109.1s


Chunk 61–70/100 hoàn thành trong 110.3s


Chunk 71–80/100 hoàn thành trong 109.3s


Chunk 81–90/100 hoàn thành trong 118.2s


Chunk 91–100/100 hoàn thành trong 107.5s
Đã lưu: D:\Github\mcs-train-content-model\results\runs\qwen3.5-2b-marketing_20260514T085930Z.csv


In [26]:
summary_cols = ["faithfulness_combined", "expansion_combined", "vibe_combined"]
means = df_scores[summary_cols].mean().to_dict()
overall = float(np.mean([means[c] for c in summary_cols]))

now = datetime.now(timezone.utc).isoformat()
summary_row = {
    "run_id": RUN_ID,
    "judge_backend": JUDGE_BACKEND,
    "judge_model": JUDGE_MODEL,
    "local_llm_backend": LOCAL_LLM_BACKEND,
    "local_model_id": LOCAL_MODEL_ID,
    "local_openai_base": LOCAL_OPENAI_BASE,
    "dataset_file": str(DATASET_PATH.relative_to(ROOT)),
    "n_cases": len(cases),
    "csv_file": str(csv_path.relative_to(ROOT)),
    "faithfulness_combined": means["faithfulness_combined"],
    "expansion_combined": means["expansion_combined"],
    "vibe_combined": means["vibe_combined"],
    "overall_mean": overall,
    "last_updated_utc": now,
}

new_df = pd.DataFrame([summary_row])
if SUMMARY_PATH.is_file():
    prev_df = pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig")
    summary_out = pd.concat([prev_df, new_df], ignore_index=True)
else:
    summary_out = new_df
summary_out.to_csv(SUMMARY_PATH, index=False, encoding="utf-8-sig")

print(f"Đã cập nhật tổng hợp: {SUMMARY_PATH}")
pd.DataFrame([summary_row]).T

Đã cập nhật tổng hợp: D:\Github\mcs-train-content-model\results\high_quality_mock_cases_100.eval_summary.csv


,0
run_id,20260514T085930Z
judge_backend,azure
judge_model,md-gpt-5.4-mini
local_llm_backend,ollama
local_model_id,qwen3.5-2b-marketing
local_openai_base,http://192.168.92.26:11434/v1
dataset_file,dataset\high_quality_mock_cases_100.json
n_cases,100
csv_file,results\runs\qwen3.5-2b-marketing_20260514T085...
faithfulness_combined,0.993694


In [27]:
# Bảng so sánh nhanh theo model (từ file tổng hợp CSV)
pd.read_csv(SUMMARY_PATH, encoding="utf-8-sig").sort_values(["local_model_id", "run_id"])

,run_id,judge_backend,judge_model,local_llm_backend,local_model_id,local_openai_base,dataset_file,n_cases,csv_file,faithfulness_combined,expansion_combined,vibe_combined,overall_mean,last_updated_utc
8,20260429T065251Z,azure,md-gpt-5.4-mini,ollama,Qwen3.5-2B_fine_tune_marketing_social_media,http://192.168.92.26:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\Qwen3.5-2B_fine_tune_marketing_so...,0.987570,0.8362,0.6668,0.830190,2026-04-29T07:09:45.139154+00:00
9,20260504T075223Z,azure,md-gpt-5.4-mini,ollama,Qwen3.5-4B_fine_tune_marketing_social_media,http://192.168.92.26:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\Qwen3.5-4B_fine_tune_marketing_so...,0.995093,0.8704,0.6759,0.847131,2026-05-04T08:10:58.108373+00:00
7,20260421T091834Z,azure,md-gpt-5.4-mini,ollama,gemma4:26b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\gemma4_26b_20260421T091834Z.csv,0.994179,0.8461,0.6787,0.839660,2026-04-21T09:37:58.050787+00:00
5,20260421T080508Z,azure,md-gpt-5.4-mini,ollama,gemma4:31b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\gemma4_31b_20260421T080508Z.csv,0.992146,0.8092,0.6780,0.826449,2026-04-21T08:25:03.306885+00:00
4,20260421T072714Z,azure,md-gpt-5.4-mini,ollama,gemma4:e2b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\gemma4_e2b_20260421T072714Z.csv,0.992563,0.8524,0.6668,0.837254,2026-04-21T07:46:37.488522+00:00
6,20260421T082805Z,azure,md-gpt-5.4-mini,ollama,gemma4:e4b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\gemma4_e4b_20260421T082805Z.csv,0.988058,0.8254,0.6766,0.830019,2026-04-21T08:47:38.121379+00:00
10,20260514T070814Z,azure,md-gpt-5.4-mini,ollama,qwen3.5-2b-marketing,http://192.168.92.26:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\qwen3.5-2b-marketing_20260514T070...,0.995333,0.8587,0.6829,0.845644,2026-05-14T07:28:57.774614+00:00
11,20260514T085930Z,azure,md-gpt-5.4-mini,ollama,qwen3.5-2b-marketing,http://192.168.92.26:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\qwen3.5-2b-marketing_20260514T085...,0.993694,0.8686,0.6822,0.848165,2026-05-14T09:34:55.614966+00:00
2,20260420T063825Z,azure,md-gpt-5.4-mini,ollama,qwen3.5:27b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\qwen3.5_27b_20260420T063825Z.csv,0.994998,0.8614,0.6724,0.842933,2026-04-20T06:55:55.187835+00:00
0,20260420T042242Z,azure,md-gpt-5.4-mini,ollama,qwen3.5:2b,http://192.168.92.22:11434/v1,dataset\high_quality_mock_cases_100.json,100,results\runs\qwen3.5_2b_20260420T042242Z.csv,0.995222,0.8488,0.6759,0.839974,2026-04-20T04:49:34.470011+00:00
